# GnuCash Data Exploration

This notebook demonstrates how to interact with GnuCash files using the piecash library.

## Setup

First, ensure you have a GnuCash file (SQLite or MySQL format) to work with.

In [ ]:
from sqlite3.dbapi2 import Date

from piecash import open_book

book = open_book("/path/to/project/db/mcp.gnucash", readonly=True)

# Test: List all accounts
print("=== Accounts ===")
for account in book.accounts:
    print(f"{account.name} ({account.type})")

# Test: Get recent transactions (last 5)
print("\n=== Recent Transactions ===")
all_splits = []
for account in book.accounts:
    all_splits.extend(account.splits)

# Sort by date, get last 5
recent_splits = sorted(all_splits, key=lambda s: s.transaction.post_date, reverse=True)[:5]

for split in recent_splits:
    tx = split.transaction
    print(f"{tx.post_date} - {tx.description} - {split.value}")

book.close()

The user wants to create a new transaction in a GnuCash file using the `piecash` library within a Jupyter Notebook.

Task Summary:
1. Re-open the GnuCash book in read-write mode (since the previous cell closed it and opened it as `readonly=True`).
2. Identify or create two accounts (e.g., an Expense account and a Bank account) to create a balanced transaction.
3. Create a `Transaction` object with two `Split` objects.
4. Save the changes to the GnuCash file.

Note: Since I don't know the exact account names in the user's specific GnuCash file, I will fetch the first two available accounts to demonstrate the process.



In [ ]:
from piecash import open_book, Transaction, Split
from datetime import datetime
from decimal import Decimal

# Re-open the book in read-write mode (readonly=False)
book = open_book("/path/to/project/db/mcp.gnucash", readonly=False)

# Get the default currency
currency = book.default_currency

# Select two accounts to create a transaction between
# Using existing accounts from the book
acc_asset = book.accounts(fullname="Assets:Current Assets:Cash in Wallet")

print(acc_asset.guid)
print(acc_asset.name)
acc_expense = book.accounts(fullname="Expenses:Hobbies")

# Create a new transaction
now = datetime.now()
trans = Transaction(
    post_date=now.date(),
    enter_date=now,
    description="Test transaction from PyCharm",
    currency=currency,
    splits=[
        Split(account=acc_asset, value=Decimal("-10.00")),
        Split(account=acc_expense, value=Decimal("10.00")),
    ]
)

# Add the transaction to the book
book.add(trans)

# Save (commit) the changes
book.save()

print(f"Transaction created: {trans.description} between {acc_asset.name} and {acc_expense.name}")

book.close()


<llm-snippet-file>gnucash_exploration.ipynb</llm-snippet-file>


### Reading & Inserting Demo Data
The following cells read and display the contents of the `demo-data-1.csv` file using the `pandas` library.

In [ ]:
import pandas as pd
from datetime import timedelta
from datetime import date

# Load the CSV file located in the same directory
df = pd.read_csv(filepath_or_buffer='demo-data-1.csv', sep=',', header=0, parse_dates=['Date', 'Value Date'], encoding='utf-8', dayfirst=True)

# View the data
# print(df.head())
# print(df.info())
# print(df.describe())

# print(df['Debit'])
print(df.Debit)  # alternative

df['Debit'] = pd.to_numeric(df['Debit'], errors='coerce')
df['Credit'] = pd.to_numeric(df['Credit'], errors='coerce')

# Display the first few rows

from piecash import open_book, Transaction, Split
from datetime import datetime
from decimal import Decimal

# Re-open the book in read-write mode (readonly=False)
book = open_book("/path/to/project/db/mcp.gnucash", readonly=False)

# Get the default currency
currency = book.default_currency

# Select two accounts to create a transaction between
# Using existing accounts from the book
savings_asset_account = book.accounts(fullname="Assets:Current Assets:Savings Account")
other_account = book.accounts(fullname="Expenses:Hobbies")



def transaction_exists(_book, _post_date, _description, _account, _value):
    """Check if transaction already exists"""
    candidates = (
        _book.query(Transaction)
        .filter(Transaction.post_date >= _post_date)
        .filter(Transaction.post_date < (_post_date + timedelta(days=1)))
        .filter(Transaction.description == _description)
        .all()
    )

    print(f"Checking {_description} for {_account} with value {_value}")
    print(f"Found {len(candidates)} candidates")

    for _trans in candidates:
        for _split in _trans.splits:
            if _split.account == _account and _split.value == _value:
                return True
    return False



# Process each transaction
for idx, row in df.iterrows():
    # Parse the transaction details to determine expense category
    description = row['Details']

    # Determine amount and whether it's debit or credit
    debt_value = 0
    credit_value = 0

    if pd.notna(row['Debit']):
        debt_value = Decimal(str(row['Debit']))
        credit_value = Decimal(-debt_value)
    elif pd.notna(row['Credit']):
        credit_value = Decimal(str(row['Credit']))
        debt_value = Decimal(-credit_value)
    else:
        continue  # Skip if no amount
    print(f"{idx}: {row.Debit} {debt_value} :: {row.Credit} {credit_value}")

    # Create the transaction
    post_date=row['Date'].date()

    trans = Transaction(
        post_date=post_date,
        enter_date=datetime.now(),
        description=description,
        currency=currency,
        splits=[
            Split(account=savings_asset_account, value=debt_value),
            Split(account=other_account, value=credit_value),
        ]
    )
    #
    # In your loop:
    if transaction_exists(book, post_date, description, savings_asset_account, debt_value):
        print(f"SKIPPED (duplicate): {post_date} - {description[:50]}")
    else:
        book.add(trans)
        book.save()
        print(f"Added: {post_date} - {description[:50]}")


# Save (commit) the changes
# print(f"Transaction created: {trans.description} between {acc_asset.name} and {acc_expense.name}")

book.close()




In [ ]:
from datetime import timedelta
from datetime import date

book = open_book("/path/to/project/db/mcp.gnucash", readonly=False)

post_date = date(2026,1,1)
description = "REF 606618000108 CINEMA CITY MALL 515390 REF-606618000108 26/03/07 21.10.55"
book.query(Transaction).filter(Transaction.post_date >= post_date).filter(Transaction.post_date >= (post_date + timedelta(days=1))
).all()


In [ ]:
text = "REF 606216000105 KUNAFA HOUSE 318455 REF-606216000105 26/03/03 19.52.41 VISA ELECTRON PURCHASE"
words = [word for word in text.split() if word.isalpha()]
import re
result = re.findall(r'\b[A-Za-z]+\b', text)
result2 = list(set(result))
print(result2)


In [ ]:

from sqlalchemy.dialects import sqlite
from sqlalchemy import or_, case, func
from piecash import open_book, Transaction, Split, Account



def find_most_recent_transaction_account(_book, search_terms, _bank_account):
    filters = []
    scores = []

    for term in search_terms:
        term_lower = term.lower()
        condition = func.lower(Transaction.description).like(f'%{term_lower}%')
        filters.append(condition)
        scores.append(case([(condition, 1)], else_=0))

    relevance = sum(scores).label('relevance')

    result = (
        _book.query(Account, Transaction, relevance)
        .join(Split, Transaction.guid == Split.transaction_guid)
        .join(Account, Account.guid == Split.account_guid)
        .filter(Split.account_guid != _bank_account.guid)
        .filter(or_(*filters))
        .order_by(relevance.desc(), Transaction.post_date.desc())
        .first()
    )

    return result





book = open_book("/path/to/project/db/mcp.gnucash", readonly=True)

# Usage
bank_account = book.accounts(fullname="Assets:Current Assets:Savings Account")


# Usage - unpacking
result = find_most_recent_transaction_account(book, ["bistro"], bank_account)
if result:
    account, transaction, score = result
    print(f" • {account.name}: {transaction.description} (score: {score})")

book.close()


In [ ]:
from piecash import open_book,Transaction, Split

book = open_book("/path/to/project/db/mcp.gnucash", readonly=True)
bank_account = book.accounts(fullname="Assets:Current Assets:Savings Account")

print(bank_account.guid)

result = (
    book.query(Transaction)
    .join(Split, Transaction.guid == Split.transaction_guid)
    .filter(Split.account_guid == bank_account.guid)
 .all())

print(result)
book.close()


